In [ ]:
import os
import sys
import pandas as pd
import torch
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
import platform

In [8]:
cwd = os.getcwd()
config_path = os.getcwd()

sys.path.append(config_path)
import config

database = config.database
central_banks = config.central_banks
training_data = os.path.join(database, "Training Data")
fed_docs = config.fed_docs
ecb_docs = config.ecb_docs

# hard coded paths for now, will need to be changed for different users before deployment
fed_training_data = pd.read_csv(
    "/Users/kylenabors/Documents/Database/Testing Data/Labeled Sentiment Randomized/Fed Labeled Sentiment.csv"
)
ecb_training_data = pd.read_csv(
    "/Users/kylenabors/Documents/Database/Testing Data/Labeled Sentiment Randomized/ECB Labeled Sentiment.csv"
)

In [ ]:
platform.platform()

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using MPS: {torch.ones(1, device=device)}")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

# ignore_mismatched_sizes replaces the 3-class head with a fresh 2-class head
model = BertForSequenceClassification.from_pretrained(
    "ZiweiChen/FinBERT-FOMC", num_labels=2, ignore_mismatched_sizes=True
).to(device)

tokenizer = BertTokenizer.from_pretrained("ZiweiChen/FinBERT-FOMC")

In [ ]:
class FinBERTDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        text = self.dataframe.iloc[index]["text"]
        label = self.dataframe.iloc[index]["sentiment"]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "text": text,
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [ ]:
RANDOM_SEED = 42
MAX_LEN = 128
BATCH_SIZE = 16
VAL_SIZE = 0.15

# Combine Fed and ECB data; drop Fed-only audience column
fed = fed_training_data.drop(columns=["audience"], errors="ignore")
training_data = pd.concat([fed, ecb_training_data], ignore_index=True)

print(f"Total samples: {len(training_data)}")
print(f"Label distribution:\n{training_data['sentiment'].value_counts()}")

train_df, val_df = train_test_split(
    training_data,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=training_data["sentiment"],
)
print(f"\nTrain: {len(train_df)}  Val: {len(val_df)}")

train_dataset = FinBERTDataset(train_df, tokenizer, max_len=MAX_LEN)
val_dataset = FinBERTDataset(val_df, tokenizer, max_len=MAX_LEN)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [ ]:
EPOCHS = 3
WARMUP_STEPS = 100

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
)


def eval_model(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    return avg_loss, acc, f1


history = []
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = eval_model(model, val_loader, device)
    print(
        f"Epoch {epoch + 1}: train_loss={avg_train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}"
    )
    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
        }
    )

print("\nTraining complete.")
print(pd.DataFrame(history))

In [ ]:
# Classification report on validation set
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=["Dovish (0)", "Hawkish (1)"]))

# Save finetuned model using HuggingFace format (preferred over torch.save)
save_path = os.path.join(cwd, "finbert_finetuned")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")